# Tensor, Vector & Transformer Security Assessment Lab

Companion notebook for the AI/ML Shift-Left Security Assessment Platform. Synthetic examples only: no external model downloads, API keys or cloud services.

**Topics:** tensor integrity, embedding/vector behavior, attention and causal masking, transformer config, model artifact handling and policy gates.

Numerical anomalies are triage signals—not proof of a vulnerability.


## 1. Imports

In [ ]:
import numpy as np
from pprint import pprint
np.set_printoptions(precision=4, suppress=True)


## 2. Tensor integrity: shape, dtype, finite values and norm

In [ ]:
def tensor_report(name, tensor, max_norm=None):
    x = np.asarray(tensor)
    report = {"name": name, "shape": x.shape, "dtype": str(x.dtype),
              "finite": bool(np.isfinite(x).all()), "l2_norm": float(np.linalg.norm(x.astype(float).ravel())),
              "findings": []}
    if x.size == 0: report["findings"].append("empty tensor")
    if not report["finite"]: report["findings"].append("NaN/Inf values")
    if max_norm is not None and report["l2_norm"] > max_norm:
        report["findings"].append("norm above configured baseline threshold")
    return report

pprint(tensor_report("clean", np.array([[.1,.2],[.3,.4]], dtype=np.float32)))
pprint(tensor_report("corrupted", np.array([[.1,np.nan],[.3,.4]], dtype=np.float32)))


## 3. Embedding/vector assessment
Check dimensions, finite values, zero vectors and similarity. Similarity outliers alone do not establish poisoning.

In [ ]:
def embedding_report(vectors, expected_dim=None):
    x = np.asarray(vectors, dtype=float)
    findings=[]
    if x.ndim != 2: return {"findings":["expected 2D matrix"]}
    if expected_dim is not None and x.shape[1] != expected_dim: findings.append("dimension mismatch")
    if not np.isfinite(x).all(): findings.append("NaN/Inf embeddings")
    norms=np.linalg.norm(x,axis=1)
    zero=np.where(norms <= 1e-12)[0].tolist()
    if zero: findings.append(f"zero vectors at rows {zero}")
    return {"shape":x.shape,"norm_min":float(norms.min()),"norm_max":float(norms.max()),"findings":findings}

def cosine_matrix(vectors):
    x=np.asarray(vectors,dtype=float); n=np.linalg.norm(x,axis=1,keepdims=True)
    z=np.divide(x,n,out=np.zeros_like(x),where=n>0)
    return z @ z.T

V=np.array([[1.,0.,0.],[.99,.01,0.],[0.,1.,0.],[0.,0.,0.]])
pprint(embedding_report(V,3))
print(cosine_matrix(V))


### RAG/vector attack surfaces
- Poisoned or attacker-controlled source documents.
- Missing tenant metadata filters or authorization-before-retrieval.
- Stale vectors after source deletion or permission changes.
- Index/query embedding model-version mismatch.
- Metadata tampering, duplicate flooding and retrieval manipulation.

## 4. Attention invariants and causal masking

In [ ]:
def attention_findings(a, tol=1e-3):
    a=np.asarray(a,dtype=float); out=[]
    if not np.isfinite(a).all(): out.append("non-finite attention")
    if np.any(a < -1e-8): out.append("negative attention probabilities")
    if not np.allclose(a.sum(axis=-1),1.0,atol=tol): out.append("rows do not sum to one")
    return out

def causal_leakage(a, tol=1e-8):
    a=np.asarray(a,dtype=float)
    if a.shape[-1] != a.shape[-2]: return ["causal matrix must be square"]
    future=np.triu(a,k=1)
    return [f"future attention leakage max={future.max():.5g}"] if np.any(future>tol) else []

safe=np.array([[1.,0.,0.],[.4,.6,0.],[.2,.3,.5]])
unsafe=np.array([[.7,.3,0.],[.2,.5,.3],[.1,.2,.7]])
print("safe:",attention_findings(safe),causal_leakage(safe))
print("unsafe:",attention_findings(unsafe),causal_leakage(unsafe))


## 5. Transformer configuration checks

In [ ]:
def transformer_config_findings(c):
    out=[]
    h=c.get("hidden_size",c.get("n_embd")); heads=c.get("num_attention_heads",c.get("n_head"))
    layers=c.get("num_hidden_layers",c.get("n_layer")); ctx=c.get("max_position_embeddings",c.get("n_positions"))
    if isinstance(h,int) and isinstance(heads,int) and heads>0 and h%heads: out.append("hidden/head divisibility mismatch")
    if isinstance(layers,int) and layers<=0: out.append("invalid layer count")
    if isinstance(ctx,int) and ctx<=0: out.append("invalid context length")
    if c.get("trust_remote_code") is True: out.append("remote model code execution enabled")
    if c.get("use_cache") and not c.get("max_cache_tokens"): out.append("KV cache has no declared token budget")
    return out

good={"hidden_size":768,"num_attention_heads":12,"num_hidden_layers":12,
      "max_position_embeddings":2048,"trust_remote_code":False,"use_cache":True,"max_cache_tokens":4096}
bad={"hidden_size":770,"num_attention_heads":12,"num_hidden_layers":0,
     "max_position_embeddings":2048,"trust_remote_code":True,"use_cache":True}
print("good:",transformer_config_findings(good))
print("bad:",transformer_config_findings(bad))


## 6. Model artifact safety: hash, do not deserialize
This cell demonstrates streaming SHA-256. It deliberately avoids `torch.load`, `pickle.load` and `joblib.load` on untrusted files.

In [ ]:
import hashlib, tempfile, pathlib
def sha256_file(path, chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(chunk),b""): h.update(block)
    return h.hexdigest()

with tempfile.NamedTemporaryFile() as f:
    f.write(b"synthetic checkpoint bytes"); f.flush()
    print(sha256_file(f.name))


## 7. Shift-left policy gate

In [ ]:
findings=[
 {"id":"TRF-REMOTE-CODE","severity":"high"},
 {"id":"EMB-ZERO","severity":"medium"},
 {"id":"ATTN-FUTURE-LEAK","severity":"critical"},
]
BLOCK={"TRF-REMOTE-CODE","ATTN-FUTURE-LEAK"}
blocking=[x for x in findings if x["id"] in BLOCK]
{"decision":"BLOCK" if blocking else "PASS","blocking":blocking,"total":len(findings)}


## 8. Further work

1. Tensor lifecycle: activation and gradient distributions, NaN/Inf propagation, checkpoint/config compatibility.
2. Transformer internals: tokenizer special-token handling, RoPE scaling, position IDs, causal masks, KV-cache isolation and memory limits.
3. Vector databases: tenant-boundary tests, metadata filters, deletion propagation, poisoned-document retrieval and index provenance.
4. Model supply chain: signatures, hashes, model cards, dataset lineage, LoRA/adapters and safe artifact inspection.
5. Runtime: authorized prompt-injection tests, tool authorization, output validation, rate limits and resource budgets.

Use an isolated test environment and only assess systems you are authorized to test.
